In [1]:
import os
import sys
p = os.path.abspath('../..')
if p not in sys.path:
    sys.path.append(p)

from numcodecs import Blosc
import numpy as np
import tifffile as tiff
import matplotlib.pyplot as plt
import glob
import json
import zarr
import waveorder as wo
from recOrder.recOrder.compute.QLIPP_compute import reconstruct_QLIPP_3D, initialize_reconstructor, load_bg
from recOrder.recOrder.io.reader import MicromanagerReader
from recOrder.recOrder.io.writer import WaveorderWriter

from pycromanager.data import Dataset

In [18]:
def reconstruct_bire(position, bg_data, recon):
    
    wavelength = recon.lambda_illu*recon.n_media*1000
    
    print('Computing Birefringence...')

#     bg_data = load_bg(bg_path, height, width)

    bg_stokes = recon.Stokes_recon(bg_data)
    bg_stokes = recon.Stokes_transform(bg_stokes)
    
    stokes_data = recon.Stokes_recon(position)
    stokes_data = recon.Stokes_transform(stokes_data)

    S_image_tm = recon.Polscope_bg_correction(stokes_data, bg_stokes)
    recon_data = recon.Polarization_recon(S_image_tm)
    ret = recon_data[0] / (2 * np.pi) * wavelength
    
    return ret

## Gather Data Paths

In [2]:
data_dir = '/gpfs/CompMicro/rawdata/hummingbird/Cameron/2021_05_18_MouseBrain_MBP_5x/Cloz_MBP_1'
# well_name = 'Well_0'

# time_folders = glob.glob(data_dir+'*'+well_name+'*')

bg_path = '/gpfs/CompMicro/rawdata/hummingbird/Cameron/2021_05_18_MouseBrain_MBP_5x/BG/'
bg_data = load_bg(bg_path, 2048, 2048, ROI = (225, 342, 1419, 1446), cropped = True)

## Initialize Reconstruction

In [38]:
## Set Reconstruction Parameters
image_dim = (2048,2048)
wavelength = 532
swing = 0.1
N_channel = 4
NA_obj = 0.15
NA_illu = 0.4
mag = 20
N_slices = 65
z_step = 0.25
pad_z = 0
pixel_size = 6.5
bg_option = 'local_fit'
n_media = 1


lambda_illu   = wavelength / 1000                       
N_pattern     = 1
N_defocus     = N_slices+2*pad_z
z_defocus     = -(np.r_[:N_defocus]-N_defocus//2)*z_step  
ps            = pixel_size/mag                    
cali          = True                         


if N_channel == 4:
    chi = swing
    inst_mat = np.array([[1, 0, 0, -1],
                         [1, np.sin(2 * np.pi * chi), 0, -np.cos(2 * np.pi * chi)],
                         [1, -0.5 * np.sin(2 * np.pi * chi), np.sqrt(3) * np.cos(np.pi * chi) * np.sin(np.pi * chi), \
                          -np.cos(2 * np.pi * chi)],
                         [1, -0.5 * np.sin(2 * np.pi * chi), -np.sqrt(3) / 2 * np.sin(2 * np.pi * chi), \
                          -np.cos(2 * np.pi * chi)]])

#         print(f'Instrument Matrix: \n\n{inst_mat}')
else:
    chi = swing * 2 * np.pi
    inst_mat = None

print('Initializing Reconstructor...')

recon = wo.waveorder_reconstructor.waveorder_microscopy(image_dim, lambda_illu, ps, NA_obj, NA_illu, z_defocus, chi=chi,
             n_media=n_media, cali=cali, bg_option=bg_option, 
             A_matrix=inst_mat, QLIPP_birefringence_only = True, pad_z=pad_z, 
             phase_deconv='3D',illu_mode='BF', use_gpu=False, gpu_id=0)

recon.N_channel = N_channel

Initializing Reconstructor...


## Reconstruct Batch

In [8]:
data = Dataset(data_dir)

Dataset opened          


In [10]:
data = data.as_array()

In [11]:
data.shape

(99, 5, 2048, 2048)

In [41]:
ret = reconstruct_bire(data[0,0:4].compute(), bg_data, recon)

Computing Birefringence...


In [45]:
%%time
dir_ = '/gpfs/CompMicro/projects/brainarchitecture/2021_05_18_MouseBrain_MBP_5x/Retardance_Imgs'
for pos in range(data.shape[0]):
    
    ret = reconstruct_bire(data[pos,0:4].compute(), bg_data, recon)
    
    tiff.imsave(dir_+f'/img_{pos:04d}.tif', ret)

Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
Computing Birefringence...
C

In [55]:
data_fluor = Dataset('/gpfs/CompMicro/rawdata/hummingbird/Cameron/2021_05_18_MouseBrain_MBP_5x/Cloz_MBP_Fluor_1')
data_fluor = data_fluor.as_array()

Dataset opened          


In [56]:
data_fluor.shape

(99, 2, 2048, 2048)

In [57]:
img = data_fluor[0, 0].compute()

In [59]:
dir_fluor = '/gpfs/CompMicro/projects/brainarchitecture/2021_05_18_MouseBrain_MBP_5x/Fluor_Imgs'

In [61]:
for pos in range(data.shape[0]):
    img = data_fluor[pos, 0].compute()
    
    tiff.imsave(dir_fluor+f'/img_{pos:04d}.tif', img)